In [ ]:
import os, pathlib, glob, math, geopandas as gpd, cameratransform as ct
from osgeo import gdal, osr
from shapely.geometry import Point, Polygon
from PIL import Image, ExifTags

# set the base directory where the images are stored
base_dir = pathlib.Path(r"C:\Data\KODSUG26_GDL_toying\test")

### Set desired outputs
# auxfile accompanies the source image, tells GIS how to project spatially without altering image file
# warped image file is re-written using spatial information to create new projected raster file
# shapefile is a set of flat footprints approximating the camera perspectives
auxfile_output=True
warpimg_output=True
shapefile_output=True

# set subdirectories for outputs, create if necessary 
if warpimg_output: os.makedirs(warpimg_dir := base_dir.joinpath(f"warpimgs"), exist_ok = True)
if shapefile_output: os.makedirs(shpfile_dir := base_dir.joinpath(f"shapefiles"), exist_ok = True)

# pull all images into a single list from the directory
img_list = list(base_dir.glob("*.jpg"))

# this is a dictionary that I manually look up online and maintain for camera models we might use so far
sensor_width_dictionary = {"SONY ILCE-6100": 23.5,
                      "Canon EOS 5DS R": 36,
                     "NIKON D810": 35.9,
                     "Hasselblad L1D-20c": 13.2,
                     "DJI FC6520": 17.952,
                     "DJI M3E": 17.3
                          }

In [ ]:
# This section sets up metadata scraping functions and reads in a sample image for camera parameters
def pull_exif(img, remove_unresolved_tags = True, clean_make_model = True):
    with Image.open(img) as im:
        if remove_unresolved_tags == True:
            exif_pull = {ExifTags.TAGS[k]: v for k, v in im._getexif().items() if k in ExifTags.TAGS if r"\x" not in str(v)}
        else:
            exif_pull = {ExifTags.TAGS[k]: v for k, v in im._getexif().items() if k in ExifTags.TAGS}   
    if clean_make_model == True: # elimate redundancy between make and model
        if exif_pull['Make'] in exif_pull['Model']: exif_pull['Model'] = exif_pull['Model'].replace(exif_pull['Make'], '').strip()
    return exif_pull

# This function can be improved, but seems to get the job done
def pull_XMP(img):
    with Image.open(img) as im:
        XMP_pull = im.getxmp()
        if 'xmpmeta' in XMP_pull.keys():
            XMP_pull = XMP_pull['xmpmeta']
            if 'RDF' in XMP_pull.keys():
                XMP_pull = XMP_pull['RDF']
                if 'Description' in XMP_pull.keys():
                    XMP_pull = XMP_pull['Description']
    return XMP_pull

# pull exif data from first image and set up camera parameters
sample_exif_data = pull_exif(img_list[0])

# clean make model tags and combine to look up sensor width if present
make_model = f"{sample_exif_data['Make'].replace("\x00", '')} {sample_exif_data['Model'].replace("\x00", '')}"
if make_model in sensor_width_dictionary:
    sensor_width = sensor_width_dictionary[make_model]
else:
    print(f"Model name {make_model} not found in dictionary, estimating from focal lengths (may include roundoff error)")
    sensor_width = float(36*sample_exif_data['FocalLength']/sample_exif_data['FocalLengthIn35mmFilm'])

# This section creates a class that stores important photogrammetry parameters
class Camera:
    def __init__(self, name, sensorW, focalL, imageW, imageH):
        # name, focal length in mm
        self.name, self.fl = name, focalL
        # sensor width, height, diagonal in mm
        self.sw, self.sh = sensorW, (imageH/imageW)*sensorW
        self.sd = math.hypot(sensorW, self.sh)
        # image width, height, diagonal in pixels
        self.imw, self.imh, self.imd = imageW, imageH, math.hypot(imageW, imageH)
        # angle of view width, heigh, diagonal in degrees
        self.aovw, self.aovh, self.aovd = math.degrees(2*math.atan(sensorW/(2*focalL))), math.degrees(2*math.atan(self.sh/(2*focalL))), math.degrees(2*math.atan(self.sd/(2*focalL)))
        # image and sensor dimensions in single variables, for convenience
        self.imsz = self.imw, self.imh
        self.ssz = self.sw, self.sh

camobj = Camera(make_model, sensor_width, sample_exif_data['FocalLength'], sample_exif_data['ExifImageWidth'], sample_exif_data['ExifImageHeight'])

# create a rectilinear projection for the camera
rlp = ct.RectilinearProjection(focallength_mm = camobj.fl, sensor = camobj.ssz, image = camobj.imsz)

# function to pull key image-specific metadata to feed into the geotransform
def get_image_params(img, camobj, nadir_only=False):
    xmp_data = pull_XMP(img)
    pitch, yaw, roll = float(xmp_data['GimbalPitchDegree']), float(xmp_data['GimbalYawDegree']), float(xmp_data['GimbalRollDegree'])
    drone_alt = float(xmp_data['AbsoluteAltitude']) + (altitude_offset := 0)
    lat, lon = float(xmp_data['GpsLatitude']), float(xmp_data['GpsLongitude'])
    gsd = (drone_alt * camobj.sw)/(camobj.fl * camobj.imw)
    if nadir_only: return lon, lat, yaw, gsd
    else: return lon, lat, pitch, yaw, roll, drone_alt, gsd

In [ ]:
### functions used to generate and sort image coordinate pairs, used for GCPs in the warp and referencing

# sorts vertices, to help calculate midpoints along the image perimeter 
def sort_vertices(vertices):
    cx, cy = sum(p[0] for p in vertices) / len(vertices), sum(p[1] for p in vertices) / len(vertices)
    return sorted(vertices, key=lambda p: math.atan2(p[1] - cy, p[0] - cx))

# calculates midpoints between sequential pairs of vertices
def perimeter_points(subdiv=0, camobj=camobj):
    plist = [[0,0], [camobj.imw, 0], [camobj.imw, camobj.imh], [0, camobj.imh]]
    for i in range(subdiv):
        for first, second in zip(plist, plist[1:] + [plist[0]]):
            plist.append([int((first[0] + second[0]) / 2), int((first[1] + second[1]) / 2)])
        plist = sort_vertices(plist)
    return plist

# calculates a full grid of control points throughout the image to reduce stretching
# during the warp. Note that calculations increase as an exp function of subdiv value
def grid_points(subdiv=0, camobj=camobj):
    x_coords, y_coords = [0, camobj.imw], [0, camobj.imh]
    for i in range(subdiv):
        for first, second in zip(x_coords, x_coords[1:]):
            x_coords.append(int((first+second)/2))
            x_coords = sorted(x_coords)
        for first, second in zip(y_coords, y_coords[1:]):
            y_coords.append(int((first+second)/2))
            y_coords = sorted(y_coords)
    return [[x, y] for x in x_coords for y in y_coords]

In [ ]:
# generate points in a grid for each image to anchor the warp
grid_anchors = grid_points(subdiv=8)
# generate points around the perimter to draw the polygon
perimeter_anchors = perimeter_points(subdiv=1)

shapefile_list = []

sr = osr.SpatialReference()
sr.ImportFromEPSG(4326)

for img in img_list:
    try: lon, lat, pitch, yaw, roll, drone_alt, gsd = get_image_params(img, camobj)
    except Exception as e:
        print(f'Exception {e}, skipping {img.name}')
        img_list.remove(img)

    # some work in the CT package to set up the spatial perspective
    cam_orientation = ct.SpatialOrientation(drone_alt, 90+pitch, roll, yaw, 0, 0)
    cam_perspective = ct.Camera(rlp, cam_orientation)
    cam_perspective.setGPSpos(lat, lon, drone_alt)

    if (auxfile_output or warpimg_output):
        # calculate the grid of anchors for the spatial perspective
        cam_grid = [cam_perspective.gpsFromImage(j).tolist()[0:2][::-1] + [0] + j for j in grid_anchors]
        ds = gdal.Open(img)
        if ds is None: print(f"Could not open image: {img}")
    
        gcps = [gdal.GCP(*i) for i in cam_grid]
        ds.SetGCPs(gcps, sr.ExportToWkt())
        print(f'Auxfile produced for {img}')

        if warpimg_output:
            kwargs = {'format': 'JPEG', 'polynomialOrder':3, 'srcNodata': '0 0 0'}
            imgout = warpimg_dir.joinpath(f"{img.name.split(".")[0]}_geoloc.jpg")
            ds = gdal.Warp(imgout, ds, **kwargs)
            print(f'Warped photo produced at {imgout}')
        ds = None

    if shapefile_output:
        # calculate the perimeter anchors for the spatial perspective
        cam_perimeter = [cam_perspective.gpsFromImage(j)[0:2][::-1] for j in perimeter_anchors]
        # append the polygon to the total list
        shapefile_list.append(Polygon(cam_perimeter))

if shapefile_output:
    out_path = shpfile_dir.joinpath(f"{base_dir.name}_footprint.shp")
    gpd.GeoDataFrame({'imgname':img_list, 'geometry':shapefile_list}, crs=src_crs).to_file(out_path)
    print(f'Shapefile produced at {out_path}')